# Readout replay (tProc v1)

This notebook feeds recorded readout traces to the NN classifier on the board **without the DAC-to-ADC loopback**. The `nn_replay` builds put a replay player (`axis_readout_replay`) between readout 0's decimated output and the broadcaster that feeds the NN and the average and decimated buffers. In replay mode the player streams stored I/Q words from a BRAM after each readout trigger, so the NN sees the shots exactly as recorded: the board logits should equal the C model's, bit for bit, and hundreds of shots run per BRAM load.

Compared with [`../readout_mock/readout_mock.ipynb`](../readout_mock/readout_mock.ipynb), which plays the traces through the DAC and the ADC at 1/4 scale, the replay tests the NN IP and its integration (trigger, window, counting) on real data, but not the readout chain.

It is ported from the tProc v2 replay (`ml-integration-tproc-v2-2026`, `firmware/notebooks/qick_ml/readout_replay/`): the same steps, with a tProc v1 trigger program.

The notebook runs in this order:

1. Load an `nn_replay` bitstream and check the memory.
2. Align: one start delay puts BRAM word j on trace sample 100 + j.
3. Replay the 20 HLS testbench shots: the logits must equal the C simulation's.
4. Replay test-set shots (1000 by default) and score them.
5. Switch back to the live readout.

## Environment

In [ ]:
import subprocess

!git branch
!echo -n "git rev:  "; git rev-parse --short HEAD
!echo -n "git date: "; git log -1 --format=%cd

branch_name = subprocess.getoutput("git rev-parse --abbrev-ref HEAD").replace('/', '-')
print('Bitstream directory: ../216/{}'.format(branch_name))

In [ ]:
# This repo's qick_lib (it matches this branch's firmware and knows the replay player)
import sys, os
repo_qick_lib = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'qick_lib'))
if repo_qick_lib not in sys.path:
    sys.path.insert(0, repo_qick_lib)

from qick import *
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import qick
print('Using qick from:', qick.__file__)

# replay_lib first: it puts ../ (qick_ml_lib) and ../readout_mock on sys.path
import replay_lib as rp
import qick_ml_lib
import readout_mock_lib as rm

## Bitstream

An `nn_replay` build from `../216/<branch>/` (`make syn-zcu216-nn-replay[-ila]`, `make package-zcu216-nn-replay[-ila]` and `make copy-zcu216` in `firmware/`).

In [ ]:
# 'nn_replay' or 'nn_replay_ila' (with ILAs on the NN)
BUILD = 'nn_replay'

CUSTOM_BIT = '../216/{}/qick_216_{}.bit'.format(branch_name, BUILD)
CUSTOM_BIT_FULL_PATH = os.path.normpath(os.path.join(os.getcwd(), CUSTOM_BIT))
print('Custom bitstream:', CUSTOM_BIT_FULL_PATH)
if not os.path.isfile(os.path.splitext(CUSTOM_BIT_FULL_PATH)[0] + '.hwh'):
    raise FileNotFoundError('Missing HWH file next to the bitstream')

soc = QickSoc(bitfile=CUSTOM_BIT_FULL_PATH)
soccfg = soc
print(soccfg)

if not (hasattr(soccfg, 'NN_0') and hasattr(soccfg, 'axi_blk_bram_ctrl_0')):
    raise RuntimeError('This bitstream has no NN')
qick_ml_lib.set_soccfg(soccfg)
RO_CH = 0        # the NN and the replay player sit on readout 0

rb = rp.ReplayBuffer(soc)
print(rb)
mem = rm.memory_report('after loading the bitstream')

## Alignment

The player starts after each readout trigger; `start_delay` sets when its first word arrives. `rp.align` replays a ramp, finds where word 0 lands in the averager's trace, and sets the delay that puts BRAM word j on trace sample 100 + j, the NN window (`WINDOW_OFFSET = 95`). It checks the whole ramp sample for sample on every capture and takes the most frequent landing sample: the averager synchronizes the tProc trigger on its own, so some captures land one sample off (the NN and the player share one synchronized trigger and stay locked). The delay is fixed for a given build, so this only has to run once per bitstream load.

In [ ]:
START_DELAY = rp.align(rb, soc, first=rp.WINDOW_START, length=rp.WINDOW_SIZE, ro_ch=RO_CH)

# The trace with the ramp on samples 100-499
tr = rp.capture(soc, RO_CH, ro_len=600 / rp.F_RO)
plt.figure(figsize=(10, 3))
plt.plot(tr[:, 0], label='I'); plt.plot(tr[:, 1], label='Q')
plt.axvline(100, ls='--', color='red'); plt.axvline(500, ls='--', color='red', label='NN window')
plt.xlabel('Trace sample'); plt.legend(); plt.grid(True); plt.title('Replayed ramp')
plt.show()

## HLS testbench shots

The 20 test shots of the NN's HLS testbench (samples 100-499 of the first and last 10 test-set shots), with the logits of the C simulation. Replayed bit for bit, the board must give exactly those logits.

In [ ]:
tb = rm.load_dataset(None)
tb_idx = np.arange(len(tb))
tb_win = np.stack([tb.window(i) for i in tb_idx])
res_tb = rp.run_shots(rb, soc, tb_win, START_DELAY, progress=False)
ref = tb.ref_logits
equal = res_tb['logits'] == ref
print('NN predictions: {} of {}; logits equal to the C simulation: {} of {}'.format(
    res_tb['count'], len(tb), int(equal.sum()), len(tb)))
for i in tb_idx:
    print('{:3d} {:10.0f} {:10.0f} {}'.format(i, ref[i], res_tb['logits'][i], '' if equal[i] else '<<< differs'))

## Test set

Shots from the test set copied to the board (see [`../readout_mock/README.md`](../readout_mock/README.md#dataset)): `N_SHOTS` balanced random shots (half per class, `SEED`), or all 100,000 with `N_SHOTS = None`. They go in batches that fill the BRAM (163 shots of 400 samples).

In [ ]:
DATA_DIR = '/home/xilinx/data/20240528/000_770'
X_NAME = 'X_test_000_770_int16.npy'
N_SHOTS = 1000           # balanced random subset (half per class); None: all shots
SEED = 0

dataset = rm.load_dataset(DATA_DIR, x_name=X_NAME)
idx = rm.select_shots(dataset, N_SHOTS, SEED)
print('{}: replaying {} shots'.format(dataset.name, len(idx)))
mem = rm.memory_report('after loading the traces')

In [ ]:
%%time
win = np.stack([dataset.window(i) for i in idx])
res = rp.run_shots(rb, soc, win, START_DELAY)
res['idx'] = idx
print('NN predictions: {} (expected {}), {:.3f} s per shot'.format(res['count'], len(idx), res['seconds'] / len(idx)))
mem = rm.memory_report('after the run')

In [ ]:
s = rm.score(dataset, res)
print('Shots: {}'.format(s['n']))
print('               board     C model (100,000 shots)')
print('accuracy     {:7.3%}   {:7.3%}'.format(s['accuracy'], rm.C_MODEL_ACCURACY))
print('fidelity     {:7.3%}   {:7.3%}'.format(s['fidelity'], rm.C_MODEL_FIDELITY))
print('ground       {:7.3%}'.format(s['accuracy_ground']))
print('excited      {:7.3%}'.format(s['accuracy_excited']))

y = dataset.y[idx]
bins = np.linspace(res['logits'].min(), res['logits'].max(), 60)
plt.figure(figsize=(10, 4))
plt.hist(res['logits'][y == 0], bins, alpha=0.5, label='ground')
plt.hist(res['logits'][y == 1], bins, alpha=0.5, label='excited')
plt.axvline(0, color='k', ls='--'); plt.xlabel('Board logit'); plt.ylabel('Shots'); plt.legend(); plt.grid(True)
plt.show()

In [ ]:
# Save the run (the C model on the same shots can be compared on a host: see README.md)
import time
os.makedirs('results', exist_ok=True)
out = 'results/readout_replay_{}.npz'.format(time.strftime('%Y%m%d_%H%M%S'))
np.savez_compressed(out, dataset=dataset.name, bitfile=CUSTOM_BIT_FULL_PATH, start_delay=START_DELAY,
                    score=repr(s), **res)
print('Saved', out)

## Back to the live readout

Live mode passes the readout through (one register later than in the builds without the player), for the other notebooks.

In [ ]:
rb.set_mode(False)
print(rb)